# Automated VWAP Execution Research

This notebook is a separate execution-research layer for the VWAP Probability Band Engine.

It does **not** change the existing VWAP model, probability logic, visual overlay, live runner, or discretionary MT5 workflow.

The purpose is to research whether a rule-based execution layer can convert selected VWAP continuation setups into simulated trades.

Initial focus:

- continuation-only logic
- S-tier and A-tier setups only
- no B-tier discretionary/maybe trades
- no fresh orange entries
- no fresh red entries
- no reversal trades
- no chop/flat-overlay entries
- fixed Nasdaq point execution:
  - SL = -29 points
  - TP = +58 points
  - BE after +29 points

This notebook is research-only. It does not send orders to MT5.

## Workflow

The notebook will eventually follow this structure:

1. Import existing VWAP engine outputs.
2. Build automation-only features.
3. Classify continuation state.
4. Score trend health and band-shift quality.
5. Detect valid green reclaim / green rejection setups.
6. Simulate trade entry and management.
7. Export a trade log.
8. Review results and rejected setups.

Important principle:

The existing visual discretionary engine remains untouched.  
This notebook only adds an optional execution-research layer on top of the already-computed bands and context.

In [35]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_project_root(start_path: Path | None = None) -> Path:
    """
    Find the project root by walking upward until a .git folder is found.
    Falls back to the current working directory if .git is not found.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / ".git").exists():
            return path

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
LIVE_ARTIFACTS_DIR = PROJECT_ROOT / "live_artifacts"

print("Current working directory:", Path.cwd())
print("Detected project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Current working directory: c:\GitHub Projects\VWAP-probability-band-engine\notebooks
Detected project root: C:\GitHub Projects\VWAP-probability-band-engine
Data directory: C:\GitHub Projects\VWAP-probability-band-engine\data


## Automation Configuration

This config controls only the execution-research layer.

It does not alter:

- VWAP calculation
- sigma-band calculation
- probability calibration
- signal-generation internals
- live MT5 overlay behaviour
- existing discretionary workflow

In [36]:
AUTO_CONFIG = {
    # Notebook mode
    "mode": "fully_automated_research",
    "research_only": True,

    # Scope
    "continuation_only": True,
    "allow_longs": True,
    "allow_shorts": True,
    "allow_reversals": False,
    "allow_b_tier": False,
    "allow_fresh_orange_entries": False,
    "allow_fresh_red_entries": False,

    # Setup tiers
    "s_tier_score": 5,
    "a_tier_score": 4,
    "minimum_tradeable_score": 4,

    # Execution
    "entry_mode": "next_bar_open",
    "one_trade_at_a_time": True,

    # Fixed Nasdaq point model
    # Distances are positive in config.
    # PnL result is signed later: SL = -29, TP = +58, BE = 0.
    "stop_loss_points": 29.0,
    "take_profit_points": 58.0,
    "breakeven_trigger_points": 29.0,

    # Risk controls
    "risk_per_trade_pct": 1.0,
    "max_consecutive_losses": 2,
    "daily_profit_cap_pct": 8.0,

    # Session controls
    "session_timezone": "Europe/London",
    "session_start": "14:00",
    "no_new_trades_after": "19:00",

    # Candle quality filters
    "min_body_ratio": 0.25,
    "min_close_through_green": 1.0,
    "max_extension_from_green": 8.0,

    # Shift / trend context
    "shift_lookback": 3,
    "vwap_cross_lookback": 8,
    "max_vwap_crosses_before_chop": 2,
}

AUTO_CONFIG

{'mode': 'fully_automated_research',
 'research_only': True,
 'continuation_only': True,
 'allow_longs': True,
 'allow_shorts': True,
 'allow_reversals': False,
 'allow_b_tier': False,
 'allow_fresh_orange_entries': False,
 'allow_fresh_red_entries': False,
 's_tier_score': 5,
 'a_tier_score': 4,
 'minimum_tradeable_score': 4,
 'entry_mode': 'next_bar_open',
 'one_trade_at_a_time': True,
 'stop_loss_points': 29.0,
 'take_profit_points': 58.0,
 'breakeven_trigger_points': 29.0,
 'risk_per_trade_pct': 1.0,
 'max_consecutive_losses': 2,
 'daily_profit_cap_pct': 8.0,
 'session_timezone': 'Europe/London',
 'session_start': '14:00',
 'no_new_trades_after': '19:00',
 'min_body_ratio': 0.25,
 'min_close_through_green': 1.0,
 'max_extension_from_green': 8.0,
 'shift_lookback': 3,
 'vwap_cross_lookback': 8,
 'max_vwap_crosses_before_chop': 2}

## Execution Sign Convention

The config stores point distances as positive numbers.

For trade PnL:

- stop loss = `-29`
- breakeven = `0`
- take profit = `+58`

For price levels:

Long trade:

- stop = entry - 29
- breakeven trigger = entry + 29
- target = entry + 58

Short trade:

- stop = entry + 29
- breakeven trigger = entry - 29
- target = entry - 58

In [37]:
def build_trade_levels(entry_price: float, side: str, config: dict) -> dict:
    """
    Build fixed-point Nasdaq execution levels.

    Parameters
    ----------
    entry_price:
        Trade entry price.
    side:
        'long' or 'short'.
    config:
        AUTO_CONFIG dictionary.

    Returns
    -------
    dict
        Stop, breakeven trigger, and target prices.
    """
    side = side.lower()

    sl = config["stop_loss_points"]
    tp = config["take_profit_points"]
    be = config["breakeven_trigger_points"]

    if side == "long":
        return {
            "side": "long",
            "entry_price": entry_price,
            "stop_price": entry_price - sl,
            "breakeven_trigger_price": entry_price + be,
            "target_price": entry_price + tp,
        }

    if side == "short":
        return {
            "side": "short",
            "entry_price": entry_price,
            "stop_price": entry_price + sl,
            "breakeven_trigger_price": entry_price - be,
            "target_price": entry_price - tp,
        }

    raise ValueError("side must be 'long' or 'short'")

In [38]:
# Quick sanity check

long_example = build_trade_levels(entry_price=20000.0, side="long", config=AUTO_CONFIG)
short_example = build_trade_levels(entry_price=20000.0, side="short", config=AUTO_CONFIG)

pd.DataFrame([long_example, short_example])

,side,entry_price,stop_price,breakeven_trigger_price,target_price
0,long,20000.0,19971.0,20029.0,20058.0
1,short,20000.0,20029.0,19971.0,19942.0


## Data Loading and Column Validation

This section prepares the automated execution notebook to consume existing VWAP engine outputs.

It does not recalculate the model.

The goal is to safely check that a DataFrame contains the fields needed for the automation layer:

- OHLC data
- timestamp
- VWAP/reference line
- upper/lower green bands
- upper/lower orange bands
- upper/lower red bands

Column aliases are supported because different exports may use slightly different names.

In [39]:
REQUIRED_RAW_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
]

REQUIRED_ENGINE_OUTPUT_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
]

COLUMN_ALIASES = {
    "datetime": ["datetime", "time", "timestamp", "date", "Date", "Time", "Datetime"],
    "open": ["open", "Open", "OPEN"],
    "high": ["high", "High", "HIGH"],
    "low": ["low", "Low", "LOW"],
    "close": ["close", "Close", "CLOSE"],

    # Volume is useful later but not strictly required for execution rules
    "tick_volume": ["tick_volume", "volume", "Volume", "tickvol", "real_volume"],

    # VWAP/reference aliases
    "vwap": ["vwap", "VWAP", "reference", "ref", "reference_line"],

    # Upper bands
    "upper_green": [
        "upper_green", "upper_1", "upper_band_1", "band_1p", "band_1_plus",
        "band_1+", "1+", "z1_upper", "upper_sigma_1"
    ],
    "upper_orange": [
        "upper_orange", "upper_2", "upper_band_2", "band_2p", "band_2_plus",
        "band_2+", "2+", "z2_upper", "upper_sigma_2"
    ],
    "upper_red": [
        "upper_red", "upper_3", "upper_band_3", "band_3p", "band_3_plus",
        "band_3+", "3+", "z3_upper", "upper_sigma_3"
    ],

    # Lower bands
    "lower_green": [
        "lower_green", "lower_1", "lower_band_1", "band_1m", "band_1_minus",
        "band_1-", "1-", "z1_lower", "lower_sigma_1"
    ],
    "lower_orange": [
        "lower_orange", "lower_2", "lower_band_2", "band_2m", "band_2_minus",
        "band_2-", "2-", "z2_lower", "lower_sigma_2"
    ],
    "lower_red": [
        "lower_red", "lower_3", "lower_band_3", "band_3m", "band_3_minus",
        "band_3-", "3-", "z3_lower", "lower_sigma_3"
    ],
}

In [40]:
def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Return the first matching column from a list of candidate names.
    """
    existing = set(df.columns)

    for col in candidates:
        if col in existing:
            return col

    # Case-insensitive fallback
    lower_map = {str(col).lower(): col for col in df.columns}
    for col in candidates:
        if str(col).lower() in lower_map:
            return lower_map[str(col).lower()]

    return None


def normalise_vwap_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename common VWAP/band export column names into the standard names used
    by the automation layer.

    This function does not change model values.
    It only standardises column names.
    """
    out = df.copy()
    rename_map = {}

    for standard_name, aliases in COLUMN_ALIASES.items():
        matched_col = find_column(out, aliases)

        if matched_col is not None and matched_col != standard_name:
            rename_map[matched_col] = standard_name

    out = out.rename(columns=rename_map)

    return out

In [41]:
def validate_automation_columns(df: pd.DataFrame, required_columns: list[str] | None = None) -> None:
    """
    Validate that the DataFrame has the columns required for automated
    VWAP execution research.

    Raises
    ------
    ValueError
        If required columns are missing.
    """
    if required_columns is None:
        required_columns = REQUIRED_AUTOMATION_COLUMNS

    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required automation columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )

    print("All required automation columns are present.")

In [42]:
def prepare_raw_ohlc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw MT5 OHLC data for later VWAP engine processing.

    This validates only the raw market data columns.

    It does not calculate VWAP.
    It does not calculate probability bands.
    It does not change the existing model logic.
    """
    out = normalise_vwap_columns(df)

    validate_automation_columns(out, required_columns=REQUIRED_RAW_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    numeric_cols = ["open", "high", "low", "close"]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).reset_index(drop=True)

    # Optional MT5 fields
    for optional_col in ["tick_volume", "spread", "real_volume"]:
        if optional_col in out.columns:
            out[optional_col] = pd.to_numeric(out[optional_col], errors="coerce")

    # Basic candle features for later commits
    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out

def prepare_automation_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare an existing VWAP engine output DataFrame for automation research.

    This function expects VWAP and band columns to already exist.

    Use prepare_raw_ohlc_dataframe() for raw MT5 OHLC files.
    """
    out = normalise_vwap_columns(df)

    validate_automation_columns(out, required_columns=REQUIRED_ENGINE_OUTPUT_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
    ]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out

In [43]:
def list_candidate_data_files() -> list[Path]:
    """
    Recursively list likely CSV/Parquet files inside the project.

    This searches the whole repo so files are found even if they are not
    directly inside /data, /artifacts, or /live_artifacts.
    """
    patterns = ["*.csv", "*.parquet"]

    files = []
    for pattern in patterns:
        files.extend(PROJECT_ROOT.rglob(pattern))

    # Avoid common virtual environment/cache folders
    ignored_parts = {
        ".git",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
    }

    clean_files = [
        file for file in files
        if not any(part in ignored_parts for part in file.parts)
    ]

    return sorted(set(clean_files))


candidate_files = list_candidate_data_files()

print(f"Found {len(candidate_files)} candidate data files:")
for file in candidate_files[:50]:
    print(file.relative_to(PROJECT_ROOT))

Found 6 candidate data files:
artifacts\tables\backtest_results.parquet
artifacts\tables\prob_table_marginal.parquet
artifacts\tables\prob_table_trend.parquet
data\historical\US100_cash_M1_NY_session_1y.csv
data\historical\US100_cash_M1_NY_session_30d.csv
live_artifacts\exports\zone_probabilities.csv


## Optional: Load Existing Export

Use this section only when you have a CSV or Parquet file that already contains the VWAP/band output.

For now, this notebook does not generate new VWAP bands.  
It only prepares already-computed model output for automation research.

In [44]:
# Default test file for automation research.
# Start with 30d because it is faster while building the notebook.

PREFERRED_DATA_FILE = "US100_cash_M1_NY_session_30d.csv"

candidate_files = list_candidate_data_files()

matching_files = [
    file for file in candidate_files
    if file.name == PREFERRED_DATA_FILE
]

if not matching_files:
    print(f"Could not find {PREFERRED_DATA_FILE}.")
    print("Available candidate files:")
    for file in candidate_files:
        print(file.relative_to(PROJECT_ROOT))
else:
    DATA_FILE = matching_files[0]
    print(f"Using file: {DATA_FILE.relative_to(PROJECT_ROOT)}")

    if DATA_FILE.suffix.lower() == ".csv":
        raw_df = pd.read_csv(DATA_FILE)
    elif DATA_FILE.suffix.lower() == ".parquet":
        raw_df = pd.read_parquet(DATA_FILE)
    else:
        raise ValueError(f"Unsupported file type: {DATA_FILE.suffix}")

    print("Raw columns:")
    print(list(raw_df.columns))

    raw_ohlc_df = prepare_raw_ohlc_dataframe(raw_df)

    print(f"\nLoaded {len(raw_ohlc_df):,} raw OHLC rows from {DATA_FILE.relative_to(PROJECT_ROOT)}")
    display(raw_ohlc_df.head())

Using file: data\historical\US100_cash_M1_NY_session_30d.csv
Raw columns:
['datetime', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume']
All required automation columns are present.

Loaded 6,728 raw OHLC rows from data\historical\US100_cash_M1_NY_session_30d.csv


,datetime,open,high,low,close,tick_volume,spread,real_volume,candle_range,candle_body,body_ratio
0,2026-01-29 15:22:00,26033.45,26035.15,26027.95,26034.55,261,190,0,7.2,1.1,0.152778
1,2026-01-29 15:23:00,26034.45,26037.55,26032.15,26034.25,234,190,0,5.4,0.2,0.037037
2,2026-01-29 15:24:00,26034.45,26040.05,26034.45,26040.05,234,190,0,5.6,5.6,1.000000
3,2026-01-29 15:25:00,26040.15,26044.45,26036.25,26044.25,277,190,0,8.2,4.1,0.500000
4,2026-01-29 15:26:00,26044.55,26046.65,26039.25,26045.25,228,190,0,7.4,0.7,0.094595


## Existing VWAP Engine Integration

This section converts raw OHLC data into VWAP probability-band engine output using the existing `src/` modules.

It uses the existing project functions to compute:

- reference line
- sigma
- upper/lower bands
- z-score
- zone classification

Then it creates automation-friendly column names such as:

- `vwap`
- `upper_green`
- `upper_orange`
- `upper_red`
- `lower_green`
- `lower_orange`
- `lower_red`

In [45]:
import sys
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config
importlib.reload(src.config)

from src.config import CONFIG
from src.reference import compute_reference
from src.sigma import compute_sigma, compute_bands
from src.zones import compute_zscore, classify_zones_series

print("Loaded existing VWAP engine modules.")
print(f"Reference type: {CONFIG['reference_type']}")
print(f"Vol method: {CONFIG['vol_method']}")
print(f"Zone thresholds: {CONFIG['zone_thresholds']}")

Loaded existing VWAP engine modules.
Reference type: VWAP
Vol method: ewma
Zone thresholds: [0.5, 1.0, 2.0]


In [46]:
def build_existing_engine_output(raw_ohlc_df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Build VWAP probability-band output from raw OHLC data using existing project logic.

    This function does not modify the model.
    It only calls the existing src functions and then adds automation-friendly aliases.
    """
    df = raw_ohlc_df.copy()

    # Required by existing reference calculation
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    if "tick_volume" not in df.columns:
        df["tick_volume"] = 1.0

    df["tick_volume"] = pd.to_numeric(df["tick_volume"], errors="coerce").fillna(1.0).clip(lower=1.0)

    df["typical_price"] = (df["high"] + df["low"] + df["close"]) / 3.0
    df["session_date"] = df["datetime"].dt.date

    # Existing model logic
    df["reference"] = compute_reference(df, config)
    df["price_deviation"] = df["close"] - df["reference"]

    df["sigma"] = compute_sigma(df, config)

    bands = compute_bands(df, df["sigma"])
    df = pd.concat([df, bands], axis=1)

    df["z_score"] = compute_zscore(df)
    df["zone"] = classify_zones_series(df["z_score"], config["zone_thresholds"])

    # Automation-friendly aliases
    df["vwap"] = df["reference"]

    df["upper_green"] = df["band_1p"]
    df["upper_orange"] = df["band_2p"]
    df["upper_red"] = df["band_3p"]

    df["lower_green"] = df["band_1n"]
    df["lower_orange"] = df["band_2n"]
    df["lower_red"] = df["band_3n"]

    # Keep candle features consistent
    df["candle_range"] = df["high"] - df["low"]
    df["candle_body"] = (df["close"] - df["open"]).abs()
    df["body_ratio"] = np.where(
        df["candle_range"] > 0,
        df["candle_body"] / df["candle_range"],
        0.0,
    )

    return df

In [47]:
engine_df = build_existing_engine_output(raw_ohlc_df, CONFIG)

print(f"Built existing engine output for {len(engine_df):,} rows.")

display_cols = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "sigma",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "z_score",
    "zone",
]

display(engine_df[display_cols].tail())

✅ VWAP computed — resets per session: True
✅ Sigma computed (ewma)
   Mean sigma : 37.132500
   Min sigma  : 14.478627
   Max sigma  : 145.465266
   Floor used : 14.478627
Built existing engine output for 6,728 rows.


,datetime,open,high,low,close,vwap,sigma,upper_green,upper_orange,upper_red,lower_green,lower_orange,lower_red,z_score,zone
6723,2026-03-17 16:25:00+00:00,24785.45,24794.45,24781.55,24792.35,24643.601231,29.695952,24673.297183,24702.993135,24732.689088,24613.905278,24584.209326,24554.513374,5.009059,Z3+
6724,2026-03-17 16:26:00+00:00,24792.25,24792.55,24779.95,24782.25,24644.731992,30.005161,24674.737154,24704.742315,24734.747477,24614.726831,24584.721670,24554.716508,4.583145,Z3+
6725,2026-03-17 16:27:00+00:00,24781.95,24783.35,24771.25,24772.75,24645.856740,29.752823,24675.609563,24705.362386,24735.115209,24616.103917,24586.351094,24556.598271,4.264915,Z3+
6726,2026-03-17 16:28:00+00:00,24772.55,24775.05,24767.55,24775.05,24647.079114,29.509526,24676.588640,24706.098167,24735.607693,24617.569588,24588.060061,24558.550535,4.336596,Z3+
6727,2026-03-17 16:29:00+00:00,24774.85,24782.25,24767.05,24769.75,24648.240001,29.043532,24677.283533,24706.327065,24735.370598,24619.196468,24590.152936,24561.109404,4.183720,Z3+


In [48]:
automation_ready_df = prepare_automation_dataframe(engine_df)

print(f"Automation-ready DataFrame: {len(automation_ready_df):,} rows")
print("Required automation columns are now available.")

display(
    automation_ready_df[
        [
            "datetime",
            "close",
            "vwap",
            "upper_green",
            "upper_orange",
            "upper_red",
            "lower_green",
            "lower_orange",
            "lower_red",
            "body_ratio",
        ]
    ].tail()
)

All required automation columns are present.
Automation-ready DataFrame: 6,728 rows
Required automation columns are now available.


,datetime,close,vwap,upper_green,upper_orange,upper_red,lower_green,lower_orange,lower_red,body_ratio
6723,2026-03-17 16:25:00+00:00,24792.35,24643.601231,24673.297183,24702.993135,24732.689088,24613.905278,24584.209326,24554.513374,0.534884
6724,2026-03-17 16:26:00+00:00,24782.25,24644.731992,24674.737154,24704.742315,24734.747477,24614.726831,24584.721670,24554.716508,0.793651
6725,2026-03-17 16:27:00+00:00,24772.75,24645.856740,24675.609563,24705.362386,24735.115209,24616.103917,24586.351094,24556.598271,0.760331
6726,2026-03-17 16:28:00+00:00,24775.05,24647.079114,24676.588640,24706.098167,24735.607693,24617.569588,24588.060061,24558.550535,0.333333
6727,2026-03-17 16:29:00+00:00,24769.75,24648.240001,24677.283533,24706.327065,24735.370598,24619.196468,24590.152936,24561.109404,0.335526
